In [19]:
import pandas as pd
import numpy as np

In [20]:
stocks = pd.read_csv(
    r"E:\Portfolio_Optimization_Project\data\all_adj_close.csv",
    parse_dates=["Date"],
    index_col="Date"
)
stocks

,AAPL,MSFT,GOOG
Date,,,
2020-01-02,72.716057,153.323288,68.046204
2020-01-03,72.009132,151.414154,67.712273
2020-01-06,72.582916,151.805496,69.381882
2020-01-07,72.241539,150.421371,69.338577
2020-01-08,73.403641,152.817322,69.885002
...,...,...,...
2025-04-23,204.600006,374.390015,157.720001
2025-04-24,208.369995,387.299988,161.470001
2025-04-25,209.279999,391.850006,163.850006


In [21]:
weights = np.random.random(3)
weights = weights/np.sum(weights)
number_of_symbols = len(weights)
returns = stocks.pct_change().dropna(how='all')
exp_ret = np.sum((returns.mean() *weights) * 252)
var = np.dot(weights.T,np.dot(returns.cov() * 252,weights))
exp_vol = np.sqrt(var)

sharpe_ratio = exp_ret / exp_vol
metrics_df = pd.DataFrame(data={
    'Expected Portfolio Returns': exp_ret,
    'Expected Portfolio Volatility': exp_vol,
    'Portfolio Sharpe Ratio': sharpe_ratio
}, index=[0])

print('')
print('='*80)
print('PORTFOLIO METRICS:')
print('-'*80)
print(metrics_df)
print('-'*80)
returns



PORTFOLIO METRICS:
--------------------------------------------------------------------------------
   Expected Portfolio Returns  Expected Portfolio Volatility  \
0                    0.225366                       0.289836   

   Portfolio Sharpe Ratio  
0                0.777565  
--------------------------------------------------------------------------------


,AAPL,MSFT,GOOG
Date,,,
2020-01-03,-0.009722,-0.012452,-0.004907
2020-01-06,0.007968,0.002585,0.024657
2020-01-07,-0.004703,-0.009118,-0.000624
2020-01-08,0.016086,0.015928,0.007881
2020-01-09,0.021241,0.012493,0.011044
...,...,...,...
2025-04-23,0.024332,0.020637,0.024821
2025-04-24,0.018426,0.034483,0.023776
2025-04-25,0.004367,0.011748,0.014740


In [22]:
# Initialize the components, to run a Monte Carlo Simulation.

# We will run 5000 iterations.
num_of_portfolios = 5000

# Prep an array to store the weights as they are generated, 5000 iterations for each of our 4 symbols.
all_weights = np.zeros((num_of_portfolios, number_of_symbols))

# Prep an array to store the returns as they are generated, 5000 possible return values.
ret_arr = np.zeros(num_of_portfolios)

# Prep an array to store the volatilities as they are generated, 5000 possible volatility values.
vol_arr = np.zeros(num_of_portfolios)

# Prep an array to store the sharpe ratios as they are generated, 5000 possible Sharpe Ratios.
sharpe_arr = np.zeros(num_of_portfolios)

# Start the simulations.
for ind in range(num_of_portfolios):

    # First, calculate the weights.
    weights = np.array(np.random.random(number_of_symbols))
    weights = weights / np.sum(weights)

    # Add the weights, to the `weights_arrays`.
    all_weights[ind, :] = weights

    # Calculate the expected log returns, and add them to the `returns_array`.
    ret_arr[ind] = np.sum((returns.mean() * weights) * 252)

    # Calculate the volatility, and add them to the `volatility_array`.
    vol_arr[ind] = np.sqrt(
        np.dot(weights.T, np.dot(returns.cov() * 252, weights))
    )

    # Calculate the Sharpe Ratio and Add it to the `sharpe_ratio_array`.
    sharpe_arr[ind] = ret_arr[ind]/vol_arr[ind]

# Let's create our "Master Data Frame", with the weights, the returns, the volatility, and the Sharpe Ratio
simulations_data = [ret_arr, vol_arr, sharpe_arr, all_weights]

# Create a DataFrame from it, then Transpose it so it looks like our original one.
simulations_df = pd.DataFrame(data=simulations_data).T

# Give the columns the Proper Names.
simulations_df.columns = [
    'Returns',
    'Volatility',
    'Sharpe Ratio',
    'Portfolio Weights'
]

# Make sure the data types are correct, we don't want our floats to be strings.
simulations_df = simulations_df.infer_objects()

# Print out the results.
print('')
print('='*80)
print('SIMULATIONS RESULT:')
print('-'*80)
print(simulations_df.head())
print('-'*80)


SIMULATIONS RESULT:
--------------------------------------------------------------------------------
    Returns  Volatility  Sharpe Ratio  \
0  0.233217    0.288914      0.807221   
1  0.221596    0.294301      0.752959   
2  0.224862    0.302669      0.742931   
3  0.235040    0.288658      0.814251   
4  0.229195    0.287455      0.797323   

                                   Portfolio Weights  
0  [0.37938602226170043, 0.245048792163836, 0.375...  
1  [0.016168911498938086, 0.5045207810612927, 0.4...  
2  [0.19027974395859498, 0.08143858585585692, 0.7...  
3  [0.4177141705853235, 0.29491117003282546, 0.28...  
4  [0.225536745773126, 0.4715576596111598, 0.3029...  
--------------------------------------------------------------------------------


In [23]:
# Return the Max Sharpe Ratio from the run.
max_sharpe_ratio = simulations_df.loc[simulations_df['Sharpe Ratio'].idxmax()]

# Return the Min Volatility from the run.
min_volatility = simulations_df.loc[simulations_df['Volatility'].idxmin()]

print('')
print('='*80)
print('MAX SHARPE RATIO:')
print('-'*80)
print(max_sharpe_ratio)
print('-'*80)

print('')
print('='*80)
print('MIN VOLATILITY:')
print('-'*80)
print(min_volatility)
print('-'*80)


MAX SHARPE RATIO:
--------------------------------------------------------------------------------
Returns                                                       0.238733
Volatility                                                    0.291666
Sharpe Ratio                                                  0.818513
Portfolio Weights    [0.5126592378317326, 0.3120093225742741, 0.175...
Name: 1166, dtype: object
--------------------------------------------------------------------------------

MIN VOLATILITY:
--------------------------------------------------------------------------------
Returns                                                       0.231084
Volatility                                                    0.287135
Sharpe Ratio                                                  0.804793
Portfolio Weights    [0.2868502111685013, 0.4183989203042365, 0.294...
Name: 4294, dtype: object
--------------------------------------------------------------------------------


using arithmatic return


Ryan YT

In [24]:
stocks


,AAPL,MSFT,GOOG
Date,,,
2020-01-02,72.716057,153.323288,68.046204
2020-01-03,72.009132,151.414154,67.712273
2020-01-06,72.582916,151.805496,69.381882
2020-01-07,72.241539,150.421371,69.338577
2020-01-08,73.403641,152.817322,69.885002
...,...,...,...
2025-04-23,204.600006,374.390015,157.720001
2025-04-24,208.369995,387.299988,161.470001
2025-04-25,209.279999,391.850006,163.850006


In [25]:
log_returns=stocks.pct_change()
log_returns=log_returns.dropna(how = 'all')
log_returns

,AAPL,MSFT,GOOG
Date,,,
2020-01-03,-0.009722,-0.012452,-0.004907
2020-01-06,0.007968,0.002585,0.024657
2020-01-07,-0.004703,-0.009118,-0.000624
2020-01-08,0.016086,0.015928,0.007881
2020-01-09,0.021241,0.012493,0.011044
...,...,...,...
2025-04-23,0.024332,0.020637,0.024821
2025-04-24,0.018426,0.034483,0.023776
2025-04-25,0.004367,0.011748,0.014740


In [26]:
cov_matrix = log_returns.cov()*252
cov_matrix

,AAPL,MSFT,GOOG
AAPL,0.107469,0.074409,0.069498
MSFT,0.074409,0.093757,0.074134
GOOG,0.069498,0.074134,0.106965


In [27]:
variance = weights.T @ cov_matrix @ weights
std_dev = np.sqrt(variance)

expected_return = np.sum(log_returns.mean()*weights)*252

sharpe_ratio = (expected_return - 0) / std_dev
print(expected_return)
print(std_dev)
print(sharpe_ratio)

0.2334279097557358
0.2882928818479473
0.8096901604350099


In [1]:
import matplotlib.pyplot as plt

# Scatter plot of every simulation
simulations_df.plot.scatter(
    x='Volatility', 
    y='Returns', 
    c='Sharpe Ratio',      # optional: color by Sharpe
    cmap='viridis', 
    alpha=0.6,
    figsize=(10,6)
)
plt.xlabel('Annualized Volatility (Risk)')
plt.ylabel('Annualized Expected Return')
plt.title('Risk–Return Tradeoff of Simulated Portfolios')
plt.grid(linestyle='--', alpha=0.5)
plt.show()

ModuleNotFoundError: No module named 'matplotlib'